# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

In [1]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

/tmp/ipykernel_488/1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [3]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)
275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)
277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін
291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти
340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій
344: 10 сторінок, 2455 символів — Положення про стипендію ректора
348: 22 сторінок, 25608 символів — Положення про користування гуртожитками
1219: 12 сторінок, 9159 символів — Правила внутрішнього розпорядку в студентських гуртожитках
1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності
4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи
3320: 16 сторінок, 31670 символів — ОПП F2 «Інженері

In [4]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [6]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [7]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [8]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [9]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [10]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [11]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [12]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [13]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [14]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей на питання з відповіддю: 91%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [15]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

## Відповіді на теоретичні питання 21–30

У кожному пункті: питання, експеримент (там, де він потрібен) і відповідь за його результатами.

### Питання 21
Як довго виконувалися клітинки ноутбуку 03_rag_langchain_eval? Чи були проблеми з запуском, які?

**Час виконання** (Colab, GPU T4; разом ≈ 8 хв, 484 с):

| Клітинка | Що робить | Час |
|---|---|---|
| встановлення пакетів | pip install | 22 с |
| імпорти | torch, transformers, langchain та ін. | 56 с |
| завантаження документів | 14 документів із репозиторію | 55 с |
| модель ембедингів і FAISS | Qwen3-Embedding-0.6B, 385 фрагментів | 44 с |
| reranker | BAAI/bge-reranker-v2-m3 | 29 с |
| метрики ретриву | 4 методи × 23 питання | 9 с |
| генеративна модель | Qwen3-4B-Instruct-2507 | 148 с |
| демо ланцюжка | одне питання | 4 с |
| оцінювання | 26 відповідей і 26 вердиктів судді | 117 с |

Решта клітинок (розбиття, BM25, RRF, бенчмарк, аналіз відмов) виконуються менше ніж за секунду. Найдовше завантажується 4B-модель (≈ 30 % загального часу), а потім триває оцінювання: на кожне питання потрібні два виклики моделі, відповідь і вердикт судді.

**Проблеми з запуском.** Помилок не було. Після pip install з'явилося попередження про конфлікт залежностей: google-colab 1.0.0 вимагає requests 2.32.4, а встановлено 2.34.2. Це лише попередження, потрібні пакети встановилися, на результати воно не вплинуло. Під час оцінювання transformers попередив, що пайплайн викликається послідовно на GPU; це порада щодо швидкодії, не помилка. З практичного: Colab обмежує кількість одночасних сеансів, тому три ноутбуки запускали по черзі, завершуючи попередній сеанс.

### Питання 22
Чим RecursiveCharacterTextSplitter (розділ 2) відрізняється від розбиття за реченнями з демо 2? Для чого задано параметр keep_separator="end"?

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample = ("Оплата за проживання вноситься до 1 вересня. Мешканці зобов'язані дотримуватися тиші; "
          "гості можуть перебувати до 22:00. Порушники притягуються до відповідальності.")

for mode in ["end", "start", False]:
    demo_splitter = RecursiveCharacterTextSplitter(
        chunk_size=70, chunk_overlap=0, separators=[". ", "; ", " ", ""], keep_separator=mode)
    print("keep_separator:", repr(mode))
    for part in demo_splitter.split_text(sample):
        print("   |", repr(part))

# Як це виглядає на справжніх фрагментах: на чому закінчуються фрагменти
texts = [ch.page_content for ch in chunks]
ends_clean = np.mean([t.rstrip().endswith((".", ";", "!", "?", ":")) for t in texts])
print(f"\nФрагментів: {len(texts)}; закінчуються розділовим знаком: {ends_clean:.0%}")
print("Приклад початку/кінця:", repr(texts[7][:60]), "...", repr(texts[7][-60:]))

keep_separator: 'end'
   | 'Оплата за проживання вноситься до 1 вересня.'
   | "Мешканці зобов'язані дотримуватися тиші;"
   | 'гості можуть перебувати до 22:00.'
   | 'Порушники притягуються до відповідальності.'
keep_separator: 'start'
   | 'Оплата за проживання вноситься до 1 вересня'
   | ". Мешканці зобов'язані дотримуватися тиші"
   | '; гості можуть перебувати до 22:00'
   | '. Порушники притягуються до відповідальності.'
keep_separator: False
   | 'Оплата за проживання вноситься до 1 вересня'
   | "Мешканці зобов'язані дотримуватися тиші"
   | 'гості можуть перебувати до 22:00'
   | 'Порушники притягуються до відповідальності.'

Фрагментів: 385; закінчуються розділовим знаком: 93%
Приклад початку/кінця: 'Положення про оцінювання результатів навчання (чинна версія ' ... 'ладач обирає із запропонованого діапазону значень (табл. 2).'


**Відмінність.** RecursiveCharacterTextSplitter працює за ієрархією роздільників: спершу ділить текст за першим із списку (тут ". ", тобто за реченням), потім зливає сусідні шматки, поки не набереться chunk_size = 1000 символів, і лише якщо шматок усе ще задовгий, переходить до дрібніших роздільників ("; ", потім пробіл, потім окремі символи). Перекриття (200 символів) береться з кінця попереднього фрагмента. Розбиття за реченнями з демо 2 цілісне: речення є неподільною одиницею, а фрагмент складається з цілих речень. Рекурсивний сплітер у принципі теж ріже по реченнях, але гарантує ліміт розміру, навіть якщо речення (чи пункт без крапок) дуже довге. Тому ми отримали 385 фрагментів проти 410 у демо 2 і 93 % фрагментів, що закінчуються розділовим знаком (для фіксованих 1000/200 з демо 2 це було лише 2 %).

**Навіщо keep_separator="end".** За замовчуванням роздільник залишається на початку наступного шматка, або взагалі видаляється. Експеримент вище показує три режими:
- "end": крапка або крапка з комою лишається в кінці свого речення ('...до 1 вересня.'); фрагмент виглядає як завершена думка;
- "start": наступний фрагмент починається з ". " або "; ", попередній лишається без розділового знака;
- False: знаки зникають зовсім ('...до 22:00' без крапки), і межі речень не видно ні моделі ембедингів, ні генератору.

Режим "end" зберігає пунктуацію там, де вона природна, тому ембединг і LLM бачать завершені речення, а цитати виглядають охайно.

### Питання 23
Чому у розділі 3 для FAISS обрано DistanceStrategy.MAX_INNER_PRODUCT? Чи можна було б так зробити, якби ембединги не були нормалізовані?

In [22]:
rng = np.random.default_rng(0)
D = np.array(embeddings.embed_documents(texts), dtype=np.float32)
q = np.array(embeddings.embed_query("Чи можна тримати в гуртожитку домашню тварину?"), dtype=np.float32)

norms = np.linalg.norm(D, axis=1)
print(f"Довжини векторів фрагментів: min {norms.min():.4f}, max {norms.max():.4f}; довжина вектора питання: {np.linalg.norm(q):.4f}")

# Імітуємо ненормалізовані ембединги: той самий напрямок, але різна довжина
scale = rng.uniform(0.5, 3.0, size=len(D)).astype(np.float32)
D_raw = D * scale[:, None]

cos_norm = D @ q                                                   # нормалізовані: скалярний добуток = косинус
dot_raw = D_raw @ q                                                # скалярний добуток для ненормалізованих
cos_raw = dot_raw / (np.linalg.norm(D_raw, axis=1) * np.linalg.norm(q))   # косинус для ненормалізованих

def top5(scores):
    return [(int(i), chunks[int(i)].metadata["doc_id"]) for i in np.argsort(-scores)[:5]]

print("top-5 (індекс, документ), нормалізовані, скалярний добуток:", top5(cos_norm))
print("top-5, ненормалізовані, косинус:                        ", top5(cos_raw))
print("top-5, ненормалізовані, скалярний добуток:              ", top5(dot_raw))
print(f"макс. різниця косинусів до і після масштабування: {np.abs(cos_norm - cos_raw).max():.5f} (лише через те, що довжини векторів 0.9998–1.0005, а не рівно 1)")

Довжини векторів фрагментів: min 0.9998, max 1.0005; довжина вектора питання: 1.0000
top-5 (індекс, документ), нормалізовані, скалярний добуток: [(144, '348'), (164, '348'), (155, '348'), (145, '348'), (159, '348')]
top-5, ненормалізовані, косинус:                         [(144, '348'), (164, '348'), (155, '348'), (145, '348'), (159, '348')]
top-5, ненормалізовані, скалярний добуток:               [(145, '348'), (163, '348'), (156, '348'), (168, '348'), (158, '348')]
макс. різниця косинусів до і після масштабування: 0.00027 (лише через те, що довжини векторів 0.9998–1.0005, а не рівно 1)


**Чому MAX_INNER_PRODUCT.** Модель Qwen3-Embedding видає нормалізовані вектори (довжини 0.9998–1.0005, перевірено вище). Для одиничних векторів скалярний добуток дорівнює косинусній подібності, а FAISS виконує пошук за скалярним добутком найшвидше (індекс IndexFlatIP, просте множення матриць). Тому ми отримуємо косинус «безкоштовно», без окремого обчислення довжин.

**Чи можна без нормалізації?** Ні, не безкарно. Скалярний добуток дорівнює cos(θ) · |a| · |b|, тож для ненормалізованих векторів на результат впливає довжина вектора фрагмента, а не лише напрямок (зміст). У експерименті ми помножили вектори на випадкові коефіцієнти 0.5–3 (напрямок і зміст не змінились): косинус залишився тим самим (різниця 0.0003, це похибка від того, що початкові довжини ≈ 1, а не рівно 1), а top-5 за скалярним добутком змінився: [144, 164, 155, 145, 159] → [145, 163, 156, 168, 158]. Довші вектори «вигравали» незалежно від змісту. Якщо вектори не нормалізовані, їх треба нормалізувати перед додаванням в індекс (L2-нормалізація) або використати косинус / L2-відстань, яка для ненормалізованих векторів теж залежить від довжини. MAX_INNER_PRODUCT допустимий без нормалізації лише тоді, коли довжина вектора навмисно несе корисний сигнал (як у деяких рекомендаційних моделях), а для текстових ембедингів це не так.

### Питання 24
Знайдіть власне питання до документів, на якому BM25 знаходить правильний фрагмент вище, ніж семантичний пошук, і питання, на якому навпаки (розділ 4). Поясніть, чому так відбувається.

In [26]:
def gold_chunk(idx):
    gold, ev = BENCHMARK[idx][1], BENCHMARK[idx][2]
    return next((ch for ch in chunks if ch.metadata["doc_id"] in gold and re.search(ev, ch.page_content)),
                next(ch for ch in chunks if ch.metadata["doc_id"] in gold))

# Власні формулювання; правильний фрагмент беремо з відповідного питання бенчмарку
OWN = [
    ("BM25 краще", "F2 бакалавр обсяг ЄКТС", 21),
    ("BM25 краще", "відсотків мінімальної стипендії у кошторисі академвідпустка", 15),
    ("вектори краще", "Скільки днів дається, щоб виселитися, якщо тебе відрахували?", 6),
    ("вектори краще", "Коли треба внести гроші за кімнату в студмістечку на осінній семестр?", 0),
]
for kind, question, idx in OWN:
    gold, ev = BENCHMARK[idx][1], BENCHMARK[idx][2]
    rv = first_hit_rank(vector_search(question, 20), gold, ev)
    rb = first_hit_rank(bm25_search(question, 20), gold, ev)
    shared = sorted(set(tokenize(question)) & set(tokenize(gold_chunk(idx).page_content)))
    print(f"[{kind}] {question}")
    print(f"    ранг правильного фрагмента: вектор {rv}, BM25 {rb}; спільні слова питання й фрагмента: {shared}\n")

[BM25 краще] F2 бакалавр обсяг ЄКТС
    ранг правильного фрагмента: вектор 6, BM25 3; спільні слова питання й фрагмента: ['f2', 'бакалавр', 'обсяг']

[BM25 краще] відсотків мінімальної стипендії у кошторисі академвідпустка
    ранг правильного фрагмента: вектор 6, BM25 1; спільні слова питання й фрагмента: ['відсотків', 'кошторисі', 'мінімальної', 'стипендії', 'у']

[вектори краще] Скільки днів дається, щоб виселитися, якщо тебе відрахували?
    ранг правильного фрагмента: вектор 1, BM25 None; спільні слова питання й фрагмента: []

[вектори краще] Коли треба внести гроші за кімнату в студмістечку на осінній семестр?
    ранг правильного фрагмента: вектор 1, BM25 7; спільні слова питання й фрагмента: ['в', 'за', 'на', 'семестр']



Результати експерименту (ранг правильного фрагмента; чим менше, тим краще):

| Питання | Вектор | BM25 |
|---|---|---|
| «F2 бакалавр обсяг ЄКТС» | 6 | **3** |
| «відсотків мінімальної стипендії у кошторисі академвідпустка» | 6 | **1** |
| «Скільки днів дається, щоб виселитися, якщо тебе відрахували?» | **1** | поза топ-20 |
| «Коли треба внести гроші за кімнату в студмістечку на осінній семестр?» | **1** | 7 |

**Коли виграє BM25.** Питання складаються з рідкісних точних слів, що буквально є у фрагменті («кошторисі», «відсотків», код «F2»). BM25 дає їм високу вагу IDF і одразу піднімає фрагмент. Ембединг усереднює зміст: «стипендія», «академвідпустка» близькі до сотень сусідніх фрагментів про стипендії (документи 340 і 344), а код «F2» чи формулювання «у кошторисі» ембединг майже не відрізняє від загальної теми, тому правильний фрагмент опиняється лише 6-м.

**Коли виграють вектори.** Питання перефразоване і майже не має спільних слів із текстом. У документі сказано «звільнити гуртожиток після відрахування», у питанні «виселитися, якщо тебе відрахували», тобто інші слова та інші форми слова. BM25 зіставляє токени точно (без лематизації), для української з її відмінюванням це критично: «відрахували» ≠ «відрахування», тому фрагмент навіть не потрапив у топ-20. Спільними залишаються лише службові слова («в», «за», «на»). Ембединг порівнює зміст, тому розуміє, що «гроші за кімнату в студмістечку» ≈ «оплата за проживання в гуртожитку». Тому методи доповнюють один одного, і гібридний пошук має сенс.

### Питання 25
Обчисліть "вручну" (за допомогою Python коду без використання допоміжних бібліотек) бал RRF для фрагмента, що стоїть на 2-й позиції у векторному пошуку й на 5-й позиції в BM25 (розділ 5). Чому RRF використовує лише позиції, а не самі оцінки методів пошуку?

In [27]:
c = 60
a, b = 1 / (c + 2), 1 / (c + 5)
print(f"RRF = 1/({c}+2) + 1/({c}+5) = {a:.6f} + {b:.6f} = {a + b:.6f}")
print(f"для порівняння: 1-ше місце лише в одному списку = {1 / (c + 1):.6f}, 1-ше в обох = {2 / (c + 1):.6f}")

# Перевірка функції rrf() на іграшкових списках: фрагмент T — 2-й у векторному, 5-й у BM25
from langchain_core.documents import Document
T = Document(page_content="T")
vec_list = [Document(page_content=x) for x in ("v1",)] + [T] + [Document(page_content=x) for x in ("v3", "v4", "v5")]
bm_list = [Document(page_content=x) for x in ("b1", "b2", "b3", "b4")] + [T]
print("перший у rrf():", rrf([vec_list, bm_list])[0].page_content)

# Те саме на справжньому питанні: ручний підрахунок і порівняння з hybrid_search
question = "Чи можна тримати в гуртожитку домашню тварину?"
v, bm_res = vector_search(question, 20), bm25_search(question, 20)
manual = {}
for lst in (v, bm_res):
    for rank, d in enumerate(lst, start=1):
        manual[d.page_content] = manual.get(d.page_content, 0) + 1 / (c + rank)
ranked = sorted(manual, key=manual.get, reverse=True)
pos_v = {d.page_content: r for r, d in enumerate(v, start=1)}
pos_b = {d.page_content: r for r, d in enumerate(bm_res, start=1)}
for t in ranked[:5]:
    print(f"  RRF {manual[t]:.5f} | вектор {pos_v.get(t)} | BM25 {pos_b.get(t)} | {t[:50]!r}")
print("збігається з hybrid_search:", [d.page_content for d in hybrid_search(question, 5)] == ranked[:5])

# Чому позиції, а не оцінки: масштаби оцінок двох методів несумісні
from rank_bm25 import BM25Okapi
bm25_index = BM25Okapi([tokenize(t) for t in texts])
bm_scores = bm25_index.get_scores(tokenize(question))
cos_scores = D @ q
print(f"косинус: від {cos_scores.min():.2f} до {cos_scores.max():.2f}; BM25: від {bm_scores.min():.2f} до {bm_scores.max():.2f}")

RRF = 1/(60+2) + 1/(60+5) = 0.016129 + 0.015385 = 0.031514
для порівняння: 1-ше місце лише в одному списку = 0.016393, 1-ше в обох = 0.032787
перший у rrf(): T
  RRF 0.03062 | вектор 2 | BM25 9 | 'Положення про користування гуртожитками. 4.6. Мешк'
  RRF 0.02938 | вектор 17 | BM25 1 | 'Положення про користування гуртожитками. Мешканці '
  RRF 0.02924 | вектор 6 | BM25 11 | 'Положення про користування гуртожитками. Мешканці '
  RRF 0.02896 | вектор 15 | BM25 4 | 'Положення про користування гуртожитками. Це є єдин'
  RRF 0.02869 | вектор 3 | BM25 18 | 'Положення про користування гуртожитками. Таку особ'
збігається з hybrid_search: True
косинус: від 0.05 до 0.57; BM25: від 0.00 до 8.44


**Ручний розрахунок** (c = 60, позиції 2 у векторному пошуку й 5 у BM25):

RRF = 1/(60+2) + 1/(60+5) = 0.016129 + 0.015385 = **0.031514**

Для порівняння, фрагмент на 1-му місці лише в одному списку отримує 1/61 = 0.016393, а на 1-му в обох — 2/61 = 0.032787. Тобто середня позиція в двох списках важить більше, ніж «лише перше місце» в одному. Функція rrf() на іграшкових списках ставить такий фрагмент першим. На справжньому питанні про домашню тварину ручний підрахунок зійшовся з hybrid_search: найкращий фрагмент мав позиції 2 у векторному й 9 в BM25 (1/62 + 1/69 = 0.03062).

**Чому лише позиції.** Оцінки двох методів несумісні за шкалою: косинус у цьому прикладі лежить у межах 0.05–0.57, а BM25 — від 0 до 8.44, причому BM25 не обмежений зверху й залежить від довжини питання і розміру корпусу. Просте додавання оцінок дозволило б BM25 «заглушити» вектори (або навпаки), а щоб їх нормалізувати, довелося б підбирати вагові коефіцієнти під кожен корпус. Позиції безрозмірні й порівнювані: «2-й» означає те саме в будь-якій системі, тож RRF не потребує калібрування. Ціна цього — втрата інформації про відрив (чи перший фрагмент випередив другий із великим запасом, чи ледь-ледь).

### Питання 26
Чому cross-encoder (розділ 6) застосовують лише до 20 кандидатів, а не до всіх фрагментів корпусу? Скільки проходів моделі знадобилося б на одне питання, щоб переранжувати весь корпус?

In [28]:
question = "Чи можна тримати в гуртожитку домашню тварину?"
cand = hybrid_search(question, 20)
pairs_20 = [(question, d.page_content) for d in cand]
pairs_all = [(question, ch.page_content) for ch in chunks]

reranker.predict(pairs_20[:4])   # прогрів, щоб не міряти першочергові витрати
t0 = time.time(); s20 = reranker.predict(pairs_20); t20 = time.time() - t0
t0 = time.time(); s_all = reranker.predict(pairs_all); t_all = time.time() - t0

print(f"20 кандидатів: {len(pairs_20)} проходів, {t20:.2f} с")
print(f"весь корпус:   {len(pairs_all)} проходів, {t_all:.2f} с (у {t_all / t20:.0f} разів довше)")
n_q = len(BENCHMARK)
print(f"для всього бенчмарку з {n_q} питань: ≈{n_q * t20:.0f} с проти ≈{n_q * t_all:.0f} с")
top_all = [chunks[i].metadata["doc_id"] for i in np.argsort(-s_all)[:5]]
top_20 = [d.metadata["doc_id"] for d in rerank_search(question, 5)]
print("топ-5 документів, reranker по всьому корпусу:  ", top_all)
print("топ-5 документів, reranker по 20 кандидатах:   ", top_20)

20 кандидатів: 20 проходів, 0.23 с
весь корпус:   385 проходів, 3.81 с (у 16 разів довше)
для всього бенчмарку з 26 питань: ≈6 с проти ≈99 с
топ-5 документів, reranker по всьому корпусу:   ['348', '1219', '348', '348', '348']
топ-5 документів, reranker по 20 кандидатах:    ['348', '348', '348', '1219', '1219']


**Причина.** Cross-encoder читає питання і фрагмент разом (один прохід на пару), тому не може заздалегідь порахувати й закешувати вектори документів, як це роблять ембединги. Його вартість росте лінійно з кількістю фрагментів і повторюється для кожного нового питання. Тому його використовують як другий етап, на невеликому наборі кандидатів, які вже відібрав швидкий пошук.

**Числа з експерименту.** Для одного питання 20 кандидатів = 20 проходів, 0.23 с. Для всього корпусу = **385 проходів** (стільки фрагментів), 3.81 с, тобто в 16 разів довше. На весь бенчмарк із 26 питань це ≈ 6 с проти ≈ 99 с; для корпусу з тисяч чи мільйонів документів переранжування всього було б нездійсненним. Втрат якості при цьому майже немає: серед 20 кандидатів гібридного пошуку правильний фрагмент був для всіх 23 питань (hit@20 = 1.00), тож reranker просто переставляє те, що вже знайдено. Переранжування всього корпусу дало б лише інший порядок у топ-5 того самого документа (348 і 1219), без видимого виграшу.

### Питання 27
Додайте до бенчмарку (розділ 7) три власні питання: одне з відповіддю, одне, що перефразовує наявне питання, і одне без відповіді в документах. Як змінилися метрики ретриву?

In [29]:
import pandas as pd

NEW = [
    # 1) нове питання з відповіддю (доказ: "Усі заходи повинні закінчуватися до 23:00 години")
    ("До котрої години в гуртожитку мають закінчуватися заходи?", ["1219"], r"закінчуватися до 23:00", "до 23:00"),
    # 2) перефраз питання «З якої до якої години в гуртожитку має бути тиша?»
    ("Коли в гуртожитку не можна шуміти?", BENCHMARK[2][1], BENCHMARK[2][2], BENCHMARK[2][3]),
    # 3) питання без відповіді в документах (структура така сама, як у наявних питань без відповіді)
    ("Скільки коштує паркомісце для студентів біля гуртожитку?",) + tuple(BENCHMARK[23][1:]),
]
new_answerable = [(q_, gold_, ev_) for q_, gold_, ev_, _ in NEW if gold_]

def metrics_for(search, items):
    ranks = [first_hit_rank(search(q_, 10), gold_, ev_) for q_, gold_, ev_ in items]
    return ranks, {"hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}

rows = []
for s in [vector_search, bm25_search, hybrid_search, rerank_search]:
    _, before = metrics_for(s, answerable)
    ranks_new, after = metrics_for(s, answerable + new_answerable)
    rows.append({"метод": s.__name__, "hit@1 до": before["hit@1"], "hit@1 після": after["hit@1"],
                 "hit@5 до": before["hit@5"], "hit@5 після": after["hit@5"],
                 "MRR до": before["MRR"], "MRR після": after["MRR"],
                 "ранги двох нових": ranks_new[-2:]})
print(pd.DataFrame(rows).round(2).to_string(index=False))

print("\nЯк на нові питання відповідає весь ланцюжок (RAG + суддя):")
for q_, gold_, ev_, ref_ in NEW:
    print(str(evaluate(q_, gold_, ref_))[:600])

        метод  hit@1 до  hit@1 після  hit@5 до  hit@5 після  MRR до  MRR після ранги двох нових
vector_search      0.61         0.60      0.87         0.84    0.74       0.72           [1, 8]
  bm25_search      0.52         0.52      0.65         0.64    0.59       0.58        [1, None]
hybrid_search      0.61         0.60      0.91         0.88    0.76       0.74           [1, 8]
rerank_search      0.83         0.84      0.96         0.96    0.90       0.90           [1, 1]

Як на нові питання відповідає весь ланцюжок (RAG + суддя):
{'питання': 'До котрої години в гуртожитку мають закінчуватися заходи?', 'правильно': True, 'цитування': True, 'документи': ['1219'], 'знайдено': ['1219', '1219', '348', '1219', '348'], 'відповідь': 'Заходи в гуртожитку повинні закінчуватися до 23:00 години [1].'}
{'питання': 'Коли в гуртожитку не можна шуміти?', 'правильно': True, 'цитування': True, 'документи': ['1219'], 'знайдено': ['1219', '348', '1219', '348', '348'], 'відповідь': 'У гуртожитку не мож

**Нові питання:**
1. З відповіддю: «До котрої години в гуртожитку мають закінчуватися заходи?» (доказ: «закінчуватися до 23:00»), знайдено на 1-му місці всіма методами.
2. Перефраз питання про тишу: «Коли в гуртожитку не можна шуміти?». Вектори й гібрид знайшли правильний фрагмент лише на 8-му місці, BM25 не знайшов у топ-10 взагалі (у питанні немає слів «тиша», «години»), а reranker повернув його на 1-ше місце.
3. Без відповіді: «Скільки коштує паркомісце для студентів біля гуртожитку?».

**Зміна метрик** (23 → 25 питань із відповіддю; третє питання без відповіді в ретриві не враховується, бо немає «правильного» фрагмента):

| Метод | hit@1 | hit@5 | MRR |
|---|---|---|---|
| vector | 0.61 → 0.60 | 0.87 → 0.84 | 0.74 → 0.72 |
| bm25 | 0.52 → 0.52 | 0.65 → 0.64 | 0.59 → 0.58 |
| hybrid | 0.61 → 0.60 | 0.91 → 0.88 | 0.76 → 0.74 |
| rerank | 0.83 → 0.84 | 0.96 → 0.96 | 0.90 → 0.90 |

Метрики змінилися мало: кожне нове питання зсуває значення приблизно на 4 п. п. (1/25). Перефраз трохи просадив усі методи, крім reranker, який він навіть підтягнув: це знову показує, що reranker найстійкіший до зміни формулювання. Повний ланцюжок на нових питаннях відповів правильно: 1-е і 2-е питання з цитуванням [1], на 3-є відмовився («У наданих документах немає відповіді»), суддя визнав усі три вердикти правильними.

### Питання 28
Порівняйте метрики ретриву для чотирьох методів пошуку (розділ 7). Чому переранжування найсильніше впливає саме на hit@1?

In [30]:
rows, all_ranks = [], {}
for s in [vector_search, bm25_search, hybrid_search, rerank_search]:
    ranks = [first_hit_rank(s(q_, 20), gold_, ev_) for q_, gold_, ev_ in answerable]
    all_ranks[s.__name__] = ranks
    rows.append({"метод": s.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "hit@20": hit_rate(ranks, 20), "MRR": mrr(ranks),
                 "ранг 1": sum(r == 1 for r in ranks),
                 "ранг 2–5": sum(r is not None and 2 <= r <= 5 for r in ranks),
                 "ранг 6–20": sum(r is not None and 6 <= r <= 20 for r in ranks),
                 "поза топ-20": sum(r is None for r in ranks)})
print(pd.DataFrame(rows).round(2).to_string(index=False))

r_h, r_r = all_ranks["hybrid_search"], all_ranks["rerank_search"]
up = sum(1 for h, r in zip(r_h, r_r) if h != 1 and r == 1)
down = sum(1 for h, r in zip(r_h, r_r) if h == 1 and r != 1)
print(f"\nПісля reranker на 1-ше місце піднялося {up} питань, а зі 1-го місця впало {down}")

        метод  hit@1  hit@5  hit@20  MRR  ранг 1  ранг 2–5  ранг 6–20  поза топ-20
vector_search   0.61   0.87    1.00 0.74      14         6          3            0
  bm25_search   0.52   0.65    0.96 0.60      12         3          7            1
hybrid_search   0.61   0.91    1.00 0.76      14         7          2            0
rerank_search   0.83   0.96    1.00 0.90      19         3          1            0

Після reranker на 1-ше місце піднялося 6 питань, а зі 1-го місця впало 1


| Метод | hit@1 | hit@5 | hit@20 | MRR |
|---|---|---|---|---|
| vector | 0.61 | 0.87 | 1.00 | 0.74 |
| bm25 | 0.52 | 0.65 | 0.96 | 0.60 |
| hybrid | 0.61 | 0.91 | 1.00 | 0.76 |
| rerank | **0.83** | **0.96** | 1.00 | **0.90** |

- BM25 найслабший: без лематизації він погано справляється з відмінками української мови (один фрагмент узагалі поза топ-20).
- Вектори й гібрид мають однаковий hit@1 = 0.61, гібрид покращує лише hit@5 (0.91): RRF об'єднує списки, але не вміє точно впорядкувати верх списку.
- Reranker піднімає hit@1 з 0.61 до 0.83 (14 → 19 питань мають правильний фрагмент на 1-му місці), а hit@5 і MRR змінюються лише трохи.

**Чому саме hit@1.** Правильний фрагмент майже завжди вже є серед кандидатів (hit@20 = 1.00 для векторів, гібриду й reranker), тож проблема не в повноті, а в порядку. Перші етапи (ембединги, BM25) оцінюють питання і фрагмент окремо, тому серед кількох схожих фрагментів одного положення (наприклад, про гуртожиток) їм важко вибрати точний. Cross-encoder читає питання і фрагмент разом і бачить деталі («за перший чи другий семестр», конкретна дата), тому ставить правильний на перше місце. З 23 питань на 1-ше місце піднялося 6, і лише 1 опустилося з нього. Reranker не може підвищити hit@20 (він лише переставляє кандидатів), зате добре «ущільнює» ранги до 1, а hit@1 найчутливіший до такого впорядкування. Це важливо для RAG: генератор найбільше довіряє фрагменту [1].

### Питання 29
У розділі 9 суддею виступає та сама модель, що генерує відповіді. Які проблеми це може спричинити? Перевірте вручну вердикти судді для пʼяти відповідей і порівняйте зі своїми.

In [31]:
ref_by_q = {q_: ref_ for q_, _, _, ref_ in BENCHMARK}
ev_df = evaluation.copy()
ev_df["еталон"] = ev_df["питання"].map(ref_by_q)
wrong = ev_df[ev_df["правильно"] == False]
others = ev_df.iloc[[9, 14, 23]]
sample = pd.concat([wrong, others]).drop_duplicates("питання").head(5)
for _, row in sample.iterrows():
    print("Питання:", row["питання"])
    print("  еталон:", row["еталон"])
    print("  відповідь:", row["відповідь"])
    print("  вердикт судді:", row["правильно"], "| цитування:", row["цитування"], "\n")

print("Додатково: чи помічає суддя явно хибні відповіді?")
q0, ref0 = BENCHMARK[0][0], BENCHMARK[0][3]
for ans_ in ["Оплату потрібно внести до 1 вересня [1].", "Оплату потрібно внести до 15 листопада [1].",
             "Оплата вноситься у вересні [1].", "У наданих документах немає відповіді."]:
    print(" ", judge(q0, ref0, ans_), "|", ans_)

Питання: Для кого встановлюється стипендія ректора?
  еталон: Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].
  вердикт судді: False | цитування: True 

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  еталон: F2.
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].
  вердикт судді: False | цитування: False 

Питання: Скільки балів потрібно для оцінки «добре» B?
  еталон: 82–89 балів.
  відповідь: Для оцінки «добре» B потрібно 82 – 89 балів [2].
  вердикт судді: True | цитування: True 

Питання: За яким показником формують рейтинг для призначення академічної стипендії?
  еталон: За середнім балом семес

**Проблеми, коли суддя і генератор — та сама модель:**
- **Спільні «сліпі плями»:** якщо модель помиляється через брак знань чи упереджене розуміння, суддя здатен повторити ту саму помилку й підтвердити хибну відповідь.
- **Самоперевага (self-preference):** LLM-судді схильні вище оцінювати тексти у власному стилі, тож оцінки можуть бути завищені.
- **Вплив параметричних знань:** суддя може визнати правильним те, що «знає» сам, навіть коли еталон каже інше (так, 121 — колишній державний код спеціальності ІПЗ, і саме його назвала модель замість F2).
- **Недетермінізм і чутливість до промпту:** вердикт залежить від формулювання JUDGE_PROMPT; результат складно повторити, а точність судді вимірювати окремо.
- Тому корисні сильніша або інша модель-суддя, кілька суддів, перевірка вибірки вручну або автоматичні перевірки (як регулярний вираз-доказ у ретриві).

**Ручна перевірка 5 вердиктів:**

| № | Питання | Вердикт судді | Мій вердикт |
|---|---|---|---|
| 1 | Для кого встановлюється стипендія ректора? | False | **True**: відповідь збігається з еталоном і лише додає деталь із того самого пункту (олімпіади, конкурси); суддя занадто суворий |
| 2 | Який шифр ІПЗ у чинній ОП? | False | False: назвала 121 замість F2, суддя правий |
| 3 | Скільки балів для оцінки «добре» B? | True | True: «82 – 89» = «82–89» |
| 4 | За яким показником формують рейтинг? | True | True: збігається з еталоном (середній бал семестрового контролю) |
| 5 | Скільки коштує обід у їдальні? | True | True: правильна відмова (відповіді в документах немає) |

Збіг у 4 з 5. Єдине розходження (№ 1) показує помилку судді в бік суворості: додаткова, але підтверджена джерелом деталь розцінена як невідповідність. Додатковий тест підтвердив, що суддя ловить явно хибні відповіді: відповідь «до 1 вересня» → True, а «до 15 листопада», «у вересні» та хибна відмова → False. На такій малій вибірці (5 відповідей і 4 синтетичні) висновки про систематичну упередженість робити не можна.

### Питання 30
Розгляньте неправильні відповіді в розділі 10 «Аналіз відмов». Для кожної визначте тип помилки: промах ретриву, помилка генерації, помилка судді чи неоднозначний еталон. Обґрунтуйте.

In [34]:
for _, row in failures.iterrows():
    idx = next(i for i, b in enumerate(BENCHMARK) if b[0] == row["питання"])
    gold_, ev_, ref_ = BENCHMARK[idx][1], BENCHMARK[idx][2], BENCHMARK[idx][3]
    top = rerank_search(row["питання"], 5)
    g = gold_chunk(idx)
    print("Питання:", row["питання"])
    print("  еталон:", ref_)
    print("  відповідь моделі:", row["відповідь"])
    print("  топ-5 після reranker:", [(d.metadata["doc_id"], d.metadata["title"][:50]) for d in top])
    print("  правильний фрагмент у топ-5:", any(d.page_content == g.page_content for d in top), "| документ:", g.metadata["title"])
    print("  фрагмент-доказ:", g.page_content[:450].replace("\n", " "))
    print()

Питання: Для кого встановлюється стипендія ректора?
  еталон: Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
  відповідь моделі: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].
  топ-5 після reranker: [('344', 'Положення про стипендію ректора'), ('340', 'Правила призначення академічних стипендій'), ('344', 'Положення про стипендію ректора'), ('340', 'Правила призначення академічних стипендій'), ('340', 'Правила призначення академічних стипендій')]
  правильний фрагмент у топ-5: True | документ: Положення про стипендію ректора
  фрагмент-доказ: Положення про стипендію ректора. ЗМІСТ 1. Загальні положення……………………………….. 3 2. Порядок призначення стипендії………………… 3 ЗМІСТ 1. Загальні положення……………………………….. 3 2. Порядок призначення стипендії………………… 3 1. Загальні положен

В аналізі відмов дві відповіді позначені як неправильні:

**1. «Для кого встановлюється стипендія ректора?»: помилка судді (разом з надто стислим еталоном).**
- Ретрив спрацював: правильний фрагмент із документа 344 стоїть у топ-5.
- Генерація теж правильна: у положенні сказано, що стипендію встановлено для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності, а також переможців олімпіад, конкурсів і змагань. Модель точно переказала фрагмент (умова про переможців олімпіад, конкурсів і змагань стоїть у тому самому реченні) і додала цитату [1].
- Суддя поставив False, бо відповідь містить більше, ніж еталон. Еталон занадто короткий (пропускає умову про переможців олімпіад), тому ця відмова є хибним спрацюванням метрики, а не помилкою системи.

**2. «Який шифр спеціальності ІПЗ у чинній освітній програмі?»: промах ретриву (з помилкою через застарілий контекст).**
- Фрагмент-доказ F2 із ОПП-2025 (документ 3320) не потрапив у топ-5: 4 з 5 фрагментів — із попередньої редакції програми «ОПП 121», редакція 2024 року (документ 3329), тоді як чинна редакція 2025 року (документ 3320) використовує код F2. У бенчмарку цей фрагмент мав найгірший ранг: 8 після reranker і 9 у векторному пошуку.
- Генерація була вірною за наданим контекстом: модель відповіла «121 [2]», спираючись на фрагмент, який їй дали. Але вона не мала шансів побачити, що чинна версія називає шифр F2, і не врахувала слово «чинній» (метаданих про версію в контексті немає).
- Суддя правий (False, цитування False).
- Причина системна: два документи про одну програму суперечать один одному (стара й нова версія). Виправлення: додати до метаданих рік/версію й фільтрувати чи пріоритизувати нову версію або виключити застарілі документи з індексу.

Помилок генерації «у чистому вигляді» (коли правильний контекст був, а модель його проігнорувала) не знайдено; суддя помилився в одній відповіді з 26; неоднозначним можна вважати еталон у першому випадку.